# Using the Average Surrogate Model to Make Pseudodata

## (1): Import Libraries:

In [ ]:
import datetime
import os 

import numpy as np
import pandas as pd
import gepard as g
from gepard.fits import th_KM15
import matplotlib.pyplot as plt

## (2): Matplotlib rcparams Configuration:

In [ ]:
plt.rcParams.update({
    "text.usetex": True, "font.family": "serif",
})
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['xtick.major.size'] = 8.5
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['xtick.minor.size'] = 2.5
plt.rcParams['xtick.minor.width'] = 0.5
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['xtick.top'] = True
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['ytick.major.size'] = 8.5
plt.rcParams['ytick.major.width'] = 0.5
plt.rcParams['ytick.minor.size'] = 2.5
plt.rcParams['ytick.minor.width'] = 0.5
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['savefig.dpi'] = 300

In [ ]:
# just choose a number here
representative_replica_index = 2

single_surrogate_surface_data = pd.read_csv(f"./hpc/data/replica_{representative_replica_index}_smooth_predictions.csv")

print(f"[INFO]: Read {len(single_surrogate_surface_data)} total rows.")

In [ ]:
# majorly important, and it comes from: https://journals.aps.org/prl/pdf/10.1103/PhysRevLett.115.212003 (pg. 3)
single_surrogate_surface_data['k'] = 5.75

In [ ]:
single_surrogate_surface_data.columns

In [ ]:
number_of_kinematic_settings = (
    single_surrogate_surface_data
    .groupby(['k', 't', 'x_b', 'q_squared'])
    .ngroups
)

print(f"[INFO]: Found a total of {number_of_kinematic_settings} unique local fit possibilities.")

In [ ]:
single_surrogate_surface_data['kinematic_setting'] = (
    single_surrogate_surface_data
    .groupby(['k', 't', 'x_b', 'q_squared'])
    .ngroup() + 1 # to start things at 1
)

In [ ]:
_MASS_OF_PROTON_IN_GEV = 0.93827208816
EPSILON_LIMIT = 1e-5

def compute_epsilon(x_b, q_squared):
    """https://arxiv.org/pdf/hep-ph/0112108"""
    return 2. * x_b * _MASS_OF_PROTON_IN_GEV / np.sqrt(q_squared) # small eq. after Eq. (22)

def calculate_t_min_bkm10(x_b, q_squared):
    """https://arxiv.org/pdf/hep-ph/0112108"""
    epsilon = compute_epsilon(x_b, q_squared)
    exact_value = -1. * q_squared * ((2. * (1. - x_b) * (1. - np.sqrt(1. + epsilon**2)) + epsilon**2) / (4. * x_b * (1. - x_b) + epsilon**2)) # Eq. (31)
    # approximate_value = -1. * _MASS_OF_PROTON_IN_GEV**2 * x_b**2 / (1. - x_b + x_b * _MASS_OF_PROTON_IN_GEV**2 / q_squared) # Eq. (31)
    return exact_value

def compute_y(k_beam, x_b, q_squared):
    """https://arxiv.org/pdf/hep-ph/0112108"""
    epsilon = compute_epsilon(x_b, q_squared)
    return np.sqrt(q_squared) / (epsilon * k_beam)

def calculate_y_maximum_bkm10(x_b, q_squared):
    """https://arxiv.org/pdf/hep-ph/0112108"""
    epsilon = compute_epsilon(x_b, q_squared)
    exact_value = 2.*((np.sqrt(1. + epsilon**2) - 1.) / epsilon**2) # small eq. after Eq. (31)
    # approximate_value = 1. - _MASS_OF_PROTON_IN_GEV**2*x_b**2/q_squared # same line as above
    return exact_value

def compute_t_prime(t, tmin):
    return (t-tmin)

def compute_k_tilde(xb, q_squared, t, tmin, ep):
    return np.sqrt(tmin - t) * np.sqrt(((1. - xb) * np.sqrt((1. + ep**2))) + (((tmin - t) * (ep**2 + (4. * (1. - xb) * xb))) / (4. * q_squared)))

def compute_k(q_squared, y_lep, ep, k_tilde):
    return np.sqrt(((1. - y_lep + (ep**2 * y_lep**2 / 4.)) / q_squared)) * k_tilde

def compute_k_dot_delta(x_b, q_squared, t, phi_azi, ep, y_lep, k):
    """https://doi.org/10.1016/S0550-3213(02)00144-X"""
    return (
        (-1.*q_squared / (2.*y_lep*(1.+ep**2))) * 
        (1. + ((2.*k*np.cos(np.pi - phi_azi)) - ((t / q_squared)*(1.-(x_b * (2. - y_lep)) + (y_lep * ep**2 / 2.))) + (y_lep * ep**2 / 2.)))
        )

def prop_1(q_squared, kdd):
    """https://doi.org/10.1016/S0550-3213(02)00144-X"""
    return (1. + (2. * (kdd / q_squared)))

def prop_2(q_squared, t, kdd):
    """https://doi.org/10.1016/S0550-3213(02)00144-X"""
    return ((-2. * (kdd / q_squared)) + (t / q_squared))

In [ ]:
# alias recommended for readability...
df = single_surrogate_surface_data
df['y'] = compute_y(df['k'], df['x_b'], df['q_squared'])
df['y_max'] = calculate_y_maximum_bkm10(df['x_b'], df['q_squared'])
df['t_min'] = calculate_t_min_bkm10(df['x_b'], df['q_squared'])
df['epsilon'] = compute_epsilon(df['x_b'], df['q_squared'])
df['k_tilde'] = compute_k_tilde(df['x_b'], df['q_squared'], df['t'], df['t_min'], df['epsilon'])
df['big_k'] = compute_k(df['q_squared'], df['y'], df['epsilon'], df['k_tilde'])
df['kdd'] = compute_k_dot_delta(df['x_b'], df['q_squared'], df['t'], df['phi'], df['epsilon'], df['y'], df['big_k'])
df['prop_1'] = prop_1(df['q_squared'], df['kdd'])
df['prop_2'] = prop_2(df['q_squared'], df['t'], df['kdd'])
df['p1_singularity'] = df['prop_1'].abs() < EPSILON_LIMIT
df['p2_singularity'] = df['prop_2'].abs() < EPSILON_LIMIT

In [ ]:
def find_phi_crossings(group, col):
    signs = np.sign(group[col])
    sign_change = (signs != signs.shift()).fillna(False)
    return group.loc[sign_change, 'phi'].values.tolist()

bad_prop1_sets = []
bad_prop2_sets = []
crossing_info = {}

for set_id, group in df.groupby('kinematic_setting'):

    # simple numerics to tell us if there's a crossing (i.e. boolean flag)
    p1_cross = (group['prop_1'].min() * group['prop_1'].max()) <= 0
    p2_cross = (group['prop_2'].min() * group['prop_2'].max()) <= 0
    
    if p1_cross or p2_cross:
        crossing_info[set_id] = {
            'p1_phis': find_phi_crossings(group, 'prop_1') if p1_cross else [],
            'p2_phis': find_phi_crossings(group, 'prop_2') if p2_cross else []
        }
        if p1_cross: bad_prop1_sets.append(set_id)
        if p2_cross: bad_prop2_sets.append(set_id)

In [ ]:
# don't let y exceed the maximum value!
y_fail = df['y'] > df['y_max']

# key point: we compare |t| vs. |t_min|
t_fail = df['t'].abs() < df['t_min'].abs()

# now we scan for singularities in the propagators...
singular_propagator_sets = df.groupby('kinematic_setting').apply( lambda x: x['p1_singularity'].any() | x['p2_singularity'].any() )
singular_propagator_sets = singular_propagator_sets[singular_propagator_sets].index.tolist()
propagator_fail = df['kinematic_setting'].isin(singular_propagator_sets)
p1_fail = df['kinematic_setting'].isin(bad_prop1_sets)
p2_fail = df['kinematic_setting'].isin(bad_prop2_sets)

conditions = [
    (y_fail & t_fail), y_fail, t_fail,
    propagator_fail, p1_fail, p2_fail
    ]

choices = [
    "Rejected: y > ymax AND t < tmin", 
    "Rejected: y > ymax", "Rejected: t < tmin",
    "Rejected: Propagator singularity",
    "Rejected: Propagator P1 crosses zero", "Rejected: Propagator P2 crosses zero"
    ]

df['status'] = np.select(conditions, choices, default = "Accepted")

bad_sets = df.loc[df['status'] != "Accepted", 'kinematic_setting'].unique()
final_filtered_data = df[~df['kinematic_setting'].isin(bad_sets)].copy()

print(f"[INFO]: Processing logs for {df['kinematic_setting'].nunique()} sets...")

for set_id, group in df.groupby('kinematic_setting'):

    # extract the row:
    row = group.iloc[0]
    is_accepted = row['kinematic_setting'] not in bad_sets
    filename = "accepted.txt" if is_accepted else "rejected.txt"
    
    p1_min = group['prop_1'].abs().min()
    p2_min = group['prop_2'].abs().min()
    propagator_singularity_info = crossing_info.get(set_id, {'p1_phis': [], 'p2_phis': []})
    
    log_text = (
        f"[INFO]: set = {set_id}\n"
        f"[INFO]: x_b = {row['x_b']}\n"
        f"[INFO]: q_squared = {row['q_squared']}\n"
        f"[INFO]: t = {row['t']}\n"
        f"[INFO]: t_min = {row['t_min']}\n"
        f"[INFO]: y = {row['y']}\n"
        f"[INFO]: y_max = {row['y_max']}\n"
        f"[INFO]: min_abs_prop1 = {p1_min}\n"
        f"[INFO]: min_abs_prop2 = {p2_min}\n"
        f"[INFO]: p1_cross_phi = {propagator_singularity_info['p1_phis']}\n"
        f"[INFO]: p2_cross_phi = {propagator_singularity_info['p2_phis']}\n"
        f"[INFO]: result = {row['status'] if not is_accepted else 'Accepted'}"
    )

    directory = f"./data/local_fit_data/kinematic_set_{set_id}"
    full_path = os.path.join(directory, filename)
    os.makedirs(directory, exist_ok = True)
    
    with open(full_path, "w") as log_file:
        log_file.write(log_text)

print(f"[INFO]: Round two of scrubbing complete.")
print(f"[INFO]: Sets removed: {len(bad_sets)}")
print(f"[INFO]: Total number of bad sets: {len(bad_sets)}")
print(f"[INFO]: Final unique sets remaining: {final_filtered_data['kinematic_setting'].nunique()}")

In [ ]:
local_fit_groups = final_filtered_data.groupby('kinematic_setting')

In [ ]:
for set_id, group in local_fit_groups:

    fixed_k = group['k'].iloc[0]
    fixed_t = group['t'].iloc[0]
    fixed_xb = group['x_b'].iloc[0]
    fixed_q2 = group['q_squared'].iloc[0]

    print(f"[INFO]: Processing local fit set {set_id}: k = {fixed_k}, t = {fixed_t}, xb = {fixed_xb}, Q^2 = {fixed_q2}")

    phi_values = group['phi'].values + np.pi
    xsec_values = group['pred_xsec'].values
    bsa_values = group['pred_bsa'].values

    current_kinematic_setting = [
        g.DataPoint(
            xB = fixed_xb, t = fixed_t, Q2 = fixed_q2, phi = phi_val,
            process = "ep2epgamma", exptype = 'fixed target',
            in1energy = fixed_k, in1charge = -1, in1polarization = +1,
            observable = 'XS',
            fname = 'Trento')
        for phi_val in phi_values
    ]

    real_h_values = [th_KM15.ReH(datapoint) for datapoint in current_kinematic_setting]
    imag_h_values = [th_KM15.ImH(datapoint) for datapoint in current_kinematic_setting]
    real_e_values = [th_KM15.ReE(datapoint) for datapoint in current_kinematic_setting]
    imag_e_values = [th_KM15.ImE(datapoint) for datapoint in current_kinematic_setting]
    real_ht_values = [th_KM15.ReHt(datapoint) for datapoint in current_kinematic_setting]
    imag_ht_values = [th_KM15.ImHt(datapoint) for datapoint in current_kinematic_setting]
    real_et_values = [th_KM15.ReEt(datapoint) for datapoint in current_kinematic_setting]
    imag_et_values = [th_KM15.ImEt(datapoint) for datapoint in current_kinematic_setting]

    local_fit_df = pd.DataFrame({
        'set': set_id,
        'k': fixed_k,
        'q_squared': fixed_q2,
        'x_b': fixed_xb,
        't': fixed_t,
        'phi': phi_values,
        'unp_beam_unp_target_xsec': xsec_values,
        'unp_target_bsa': bsa_values,

        ####### IMPORTANT! Using KM15 VALUES! #######
        "Re[H]": real_h_values,
        "Im[H]": imag_h_values,
        "Re[E]": real_e_values,
        "Im[E]": imag_e_values,
        "Re[Ht]": real_ht_values,
        "Im[Ht]": imag_ht_values,
        "Re[Et]": real_et_values,
        "Im[Et]": imag_et_values
    })

    local_fit_df.to_csv(
        f"./data/local_fit_data/kinematic_set_{set_id}/local_fit_set_{set_id}_v1_1.csv",
        index = False
    )

In [ ]:
unique_kinematics = (
    final_filtered_data[
        ['kinematic_setting', 't', 'x_b', 'q_squared']
    ]
    .drop_duplicates()
    .sort_values('kinematic_setting')
    .reset_index(drop = True)
)

print(f"[INFO]: Obtain a total of {len(unique_kinematics)} kinematic settings.")

assert len(unique_kinematics) == local_fit_groups.ngroups, "[ASSERT]: Expected data lengths not equal..."

In [ ]:
input_space_figure = plt.figure(figsize = (9, 7), layout = "tight")
input_space_axis = input_space_figure.add_subplot(1, 1, 1, projection = "3d")

#############################################
# figure/axis augmentation details:
#############################################
axis_elevation = input_space_axis.elev # extract eleveation param
axis_azimuthal = input_space_axis.azim # extract azimuth parm

# https://matplotlib.org/stable/gallery/mplot3d/text3d.html -> for ax.text2D
input_space_axis.text2D(
    0.01, 0.03, 
    fr"elevation = {axis_elevation}, $\phi = {axis_azimuthal}^{{\circ}}$", 
    transform = input_space_axis.transAxes)
input_space_axis.text2D(
    0.01, 0.00, 
    fr"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
    transform = input_space_axis.transAxes)

#############################################
# axis plotting data:
#############################################
input_space_axis.scatter(
    unique_kinematics['x_b'], 
    unique_kinematics['q_squared'], 
    -1.0 * unique_kinematics['t'],
    color = "blue", alpha = 0.57)

#############################################
# axis labeling data:
#############################################
input_space_axis.set_title(rf"Scrubbed Kinematic Settings Space for Replica {representative_replica_index}", fontsize = 18)
input_space_axis.set_xlabel(r"$x_{\textrm{B}}$", fontsize = 16)
input_space_axis.set_ylabel(r"$Q^{2}$", fontsize = 16)
input_space_axis.set_zlabel(r"$-t$", fontsize = 16)

#############################################
# plot saving configuration:
#############################################

for extension in ["png", "eps"]:
    input_space_figure.savefig(f"./plots/scrubbed_kinematic_space_v1_1.{extension}",
        facecolor = 'white', transparent = False)
plt.close(input_space_figure)

### (X): $-t$ vs. $x_{\text{B}}$ 2D Slice Plot of Kinematic Phase Space

In [ ]:
#############################################
# figure initialization and customization
#############################################
t_vs_xb_figure, t_vs_x_axis = plt.subplots(1, 1, figsize = (8, 7), layout = "tight")

#############################################
# figure/axis augmentation details:
#############################################
t_vs_x_axis.text(
    -0.1, -0.1, 
    fr"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
    transform = t_vs_x_axis.transAxes)

#############################################
# axis plotting data:
#############################################
t_vs_x_scatter_data = t_vs_x_axis.scatter(
    unique_kinematics['x_b'],
    -1. * unique_kinematics['t'],
    c = unique_kinematics['q_squared'],
    cmap = 'plasma', s = 40, alpha = 0.57)

colorbar = t_vs_xb_figure.colorbar(t_vs_x_scatter_data)

#############################################
# axis labeling data:
#############################################
t_vs_x_axis.set_title(rf"$-t$ vs. $x_{{\textrm{{B}}}}$ Projection", fontsize = 18)
t_vs_x_axis.set_xlabel(r"$x_{\textrm{B}}$", fontsize = 16)
t_vs_x_axis.set_ylabel(r"$-t$ (GeV$^{2}$)", fontsize = 16)
colorbar.set_label(r"$Q^2$ Value", fontsize = 16)

#############################################
# axis limits
#############################################
# I got these limits from the plots in https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.115.212003
t_vs_x_axis.set_xlim(0.0, 0.6) # this is across xB
t_vs_x_axis.set_ylim(0.0, 1.0) # this is across -t
t_vs_x_axis.grid(alpha = 0.4)

#############################################
# plot saving configuration:
#############################################

for extension in ["png", "eps"]:
    t_vs_xb_figure.savefig(f"./plots/scrubbed_t_vs_xb_v1_1.{extension}",
        facecolor = 'white', transparent = False)
plt.close(t_vs_xb_figure)

### (X): $Q^{2}$ vs. $x_{\text{B}}$ 2D Slice Plot of Kinematic Phase Space

In [ ]:
#############################################
# figure initialization and customization
#############################################
qsq_vs_xb_figure, qsq_vs_x_axis = plt.subplots(1, 1, figsize = (8, 7), layout = "tight")

#############################################
# figure/axis augmentation details:
#############################################
qsq_vs_x_axis.text(
    -0.1, -0.1, 
    fr"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
    transform = qsq_vs_x_axis.transAxes)

#############################################
# axis plotting data:
#############################################
qsq_vs_x_scatter_data = qsq_vs_x_axis.scatter(
    unique_kinematics['x_b'],
    unique_kinematics['q_squared'],
    c = -1. * unique_kinematics['t'],
    cmap = 'plasma', s = 40, alpha = 0.57)

qsq_xb_colorbar = qsq_vs_xb_figure.colorbar(qsq_vs_x_scatter_data)

#############################################
# axis labeling data:
#############################################
qsq_vs_x_axis.set_title(rf"$Q^2$ vs. $x_{{\textrm{{B}}}}$ Projection", fontsize = 18)
qsq_vs_x_axis.set_xlabel(r"$x_{\textrm{B}}$", fontsize = 16)
qsq_vs_x_axis.set_ylabel(r"$Q^2$ (GeV$^{2}$)", fontsize = 16)
qsq_xb_colorbar.set_label(r"$-t$ Value", fontsize = 16)

#############################################
# axis limits
#############################################
# I got these limits from the plots in https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.115.212003
qsq_vs_x_axis.set_xlim(0.0, 0.6) # this is across xB
qsq_vs_x_axis.set_ylim(0.5, 5.0) # this is across Q^{2}
qsq_vs_x_axis.grid(alpha = 0.4)

#############################################
# plot saving configuration:
#############################################

for extension in ["png", "eps"]:
    qsq_vs_xb_figure.savefig(f"./plots/scrubbed_qsq_vs_xb_v1_1.{extension}",
        facecolor = 'white', transparent = False)
plt.close(qsq_vs_xb_figure)